# Step 3: Adjust Asset Allocations

Replicates `code/adjust_allocations.do`

**Input:** `output_python/ppd_plan_level_raw.dta` (Step 2 output)  
**Output:** `output_python/ppd_plan_clean_allocations.{dta,parquet,csv}`  
**Parity baseline:** `output/ppd_plan_clean_allocations.dta`

Strategy: parse the ~6700 `replace` statements from the `.do` file with regex, then apply programmatically.

In [1]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT = Path.cwd().parent
RAW = ROOT / 'raw'
CODE = ROOT / 'legacy' / 'code'
OUTPUT_STATA = ROOT / 'legacy' / 'output'
OUTPUT_PY = ROOT / 'output_python'
OUTPUT_PY.mkdir(exist_ok=True)

print(f'Root: {ROOT}')
print(f'Code: {CODE}')
print(f'Stata baseline: {OUTPUT_STATA}')
print(f'Python output:  {OUTPUT_PY}')

Root: /Users/Work/Desktop/Pension Research/ppd-data
Code: /Users/Work/Desktop/Pension Research/ppd-data/legacy/code
Stata baseline: /Users/Work/Desktop/Pension Research/ppd-data/legacy/output
Python output:  /Users/Work/Desktop/Pension Research/ppd-data/output_python


## 1. Parse replace statements from adjust_allocations.do

Extract every `replace COL = EXPR if CONDITIONS` line into structured tuples.

In [2]:
do_file = CODE / 'adjust_allocations.do'
raw_lines = do_file.read_text().splitlines()
print(f'Total lines in .do file: {len(raw_lines)}')

# --- Phase 1: Join multi-line statements (Stata /// continuation) ---
lines = []
buf = ''
for line in raw_lines:
    stripped = line.strip()
    if stripped.endswith('///'):
        # Strip the /// and accumulate
        buf += stripped[:-3].rstrip() + ' '
    else:
        if buf:
            buf += stripped
            lines.append(buf)
            buf = ''
        else:
            lines.append(stripped)
if buf:
    lines.append(buf)

print(f'Logical lines after joining ///: {len(lines)}')

# --- Phase 2: Parse replace statements ---
replace_re = re.compile(
    r'^replace\s+(\w+)\s*=\s*(.+?)\s+if\s+(.+)$'
)

# Parse ppd_id conditions
ppd_single_re = re.compile(r'ppd_id\s*==\s*(\d+)')

# Parse fy conditions
fy_eq_re = re.compile(r'fy\s*==\s*(\d+)')
fy_ge_re = re.compile(r'fy\s*>=\s*(\d+)')
fy_le_re = re.compile(r'fy\s*<=\s*(\d+)')

replacements = []
parse_failures = []

for i, line in enumerate(lines, 1):
    m = replace_re.match(line)
    if not m:
        continue
    
    col = m.group(1)
    rhs_str = m.group(2).strip()
    cond_str = m.group(3).strip()
    
    # Evaluate RHS
    if rhs_str == '.':
        value = np.nan
    else:
        try:
            value = float(eval(rhs_str))
        except Exception as e:
            parse_failures.append((i, line, f'RHS eval error: {e}'))
            continue
    
    # Parse conditions
    try:
        ppd_ids = [int(x) for x in ppd_single_re.findall(cond_str)]
        
        if not ppd_ids:
            parse_failures.append((i, line, 'No ppd_id found'))
            continue
        
        # Extract fy condition
        fy_eq = fy_eq_re.findall(cond_str)
        fy_ge = fy_ge_re.findall(cond_str)
        fy_le = fy_le_re.findall(cond_str)
        
        if fy_eq:
            fy_cond = ('eq', int(fy_eq[0]))
        elif fy_ge and fy_le:
            fy_cond = ('range', int(fy_ge[0]), int(fy_le[0]))
        else:
            parse_failures.append((i, line, 'No fy condition found'))
            continue
        
        replacements.append({
            'line': i,
            'col': col,
            'value': value,
            'ppd_ids': ppd_ids,
            'fy_cond': fy_cond,
        })
    except Exception as e:
        parse_failures.append((i, line, f'Condition parse error: {e}'))

print(f'\nParse report:')
print(f'  Replace statements found: {len(replacements)}')
print(f'  Parse failures: {len(parse_failures)}')
if parse_failures:
    for ln, txt, err in parse_failures[:10]:
        print(f'    Line {ln}: {err} — {txt[:100]}')

# Summary by column
from collections import Counter
col_counts = Counter(r['col'] for r in replacements)
print(f'\nReplacements per column:')
for c, n in sorted(col_counts.items()):
    print(f'  {c}: {n}')

# Unique ppd_ids
all_ppd_ids = set()
for r in replacements:
    all_ppd_ids.update(r['ppd_ids'])
print(f'\nUnique ppd_ids: {len(all_ppd_ids)}')
print(f'  {sorted(all_ppd_ids)}')

Total lines in .do file: 6702
Logical lines after joining ///: 6542

Parse report:
  Replace statements found: 3644
  Parse failures: 0

Replacements per column:
  AltMiscTotal_Actl: 141
  AltMiscTotal_Trgt: 147
  COMDTotal_Actl: 212
  COMDTotal_Trgt: 199
  CashTotal_Actl: 303
  CashTotal_Trgt: 133
  EQTotal_Actl: 247
  EQTotal_Trgt: 174
  FITotal_Actl: 319
  FITotal_Trgt: 200
  HFTotal_Actl: 182
  HFTotal_Trgt: 175
  OtherTotal_Actl: 105
  OtherTotal_Trgt: 78
  PETotal_Actl: 258
  PETotal_Trgt: 216
  RETotal_Actl: 306
  RETotal_Trgt: 249

Unique ppd_ids: 52
  [1, 2, 3, 4, 7, 8, 19, 20, 21, 22, 24, 37, 42, 45, 52, 55, 58, 61, 65, 74, 86, 92, 94, 95, 96, 97, 101, 103, 113, 114, 125, 126, 129, 130, 134, 141, 145, 146, 148, 150, 152, 178, 192, 195, 201, 206, 208, 210, 213, 215, 217, 234]


## 2. Load Step 2 output and apply corrections

In [3]:
# Load Python Step 2 output
df = pd.read_stata(OUTPUT_PY / 'ppd_plan_level_raw.dta')
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} cols')

# Ensure key columns are int for matching
df['ppd_id'] = df['ppd_id'].astype(int)
df['fy'] = df['fy'].astype(int)

# Apply all replacements in source order
rows_modified = Counter()

for r in replacements:
    col = r['col']
    val = r['value']
    ppd_ids = r['ppd_ids']
    fy_cond = r['fy_cond']
    
    # Build mask
    if len(ppd_ids) == 1:
        mask = df['ppd_id'] == ppd_ids[0]
    else:
        mask = df['ppd_id'].isin(ppd_ids)
    
    if fy_cond[0] == 'eq':
        mask = mask & (df['fy'] == fy_cond[1])
    elif fy_cond[0] == 'range':
        mask = mask & (df['fy'] >= fy_cond[1]) & (df['fy'] <= fy_cond[2])
    
    n_hit = mask.sum()
    if n_hit > 0:
        df.loc[mask, col] = val
        rows_modified[col] += n_hit

print(f'\nCorrections applied per column:')
for c, n in sorted(rows_modified.items()):
    print(f'  {c}: {n} cell updates')
print(f'\nTotal cell updates: {sum(rows_modified.values())}')

Loaded: 6268 rows x 283 cols

Corrections applied per column:
  AltMiscTotal_Actl: 141 cell updates
  AltMiscTotal_Trgt: 219 cell updates
  COMDTotal_Actl: 212 cell updates
  COMDTotal_Trgt: 271 cell updates
  CashTotal_Actl: 320 cell updates
  CashTotal_Trgt: 205 cell updates
  EQTotal_Actl: 248 cell updates
  EQTotal_Trgt: 247 cell updates
  FITotal_Actl: 335 cell updates
  FITotal_Trgt: 272 cell updates
  HFTotal_Actl: 182 cell updates
  HFTotal_Trgt: 247 cell updates
  OtherTotal_Actl: 105 cell updates
  OtherTotal_Trgt: 150 cell updates
  PETotal_Actl: 259 cell updates
  PETotal_Trgt: 289 cell updates
  RETotal_Actl: 307 cell updates
  RETotal_Trgt: 322 cell updates

Total cell updates: 4331


## 2b. Special Plan Adjustments (Internet Appendix A.1.3)

Two plans require replacing target allocation shares with actual shares:

1. **Idaho PERSI (ppd_id=31):** PPD lumps private equity into public equity for targets, setting PETotal_Trgt to zero. Fix: replace all target shares with actuals for all years.
2. **NY SLRS (ppd_id=83, FY 2001–2003):** PPD reports target shares not disclosed in annual reports. Fix: replace target shares with actuals for FY 2001–2003 only.

In [4]:
# --- Special Plan Adjustments (Internet Appendix A.1.3) ---

# Target → Actual column pairs (all 10 asset classes)
trgt_actl_pairs = [
    ('EQTotal_Trgt',      'EQTotal_Actl'),
    ('FITotal_Trgt',      'FITotal_Actl'),
    ('RETotal_Trgt',      'RETotal_Actl'),
    ('PETotal_Trgt',      'PETotal_Actl'),
    ('HFTotal_Trgt',      'HFTotal_Actl'),
    ('COMDTotal_Trgt',    'COMDTotal_Actl'),
    ('AltMiscTotal_Trgt', 'AltMiscTotal_Actl'),
    ('CashTotal_Trgt',    'CashTotal_Actl'),
    ('OtherTotal_Trgt',   'OtherTotal_Actl'),
    ('Leverage_Total_Trgt', 'Leverage_Total_Actl'),
]

# 1. Idaho PERSI (ppd_id=31): replace targets with actuals, all years
idaho_mask = df['ppd_id'] == 31
n_idaho = idaho_mask.sum()
for trgt_col, actl_col in trgt_actl_pairs:
    df.loc[idaho_mask, trgt_col] = df.loc[idaho_mask, actl_col]
print(f'Idaho PERSI (ppd_id=31): replaced targets with actuals for {n_idaho} rows')

# 2. NY SLRS (ppd_id=83): replace targets with actuals, FY 2001-2003 only
ny_mask = (df['ppd_id'] == 83) & (df['fy'].isin([2001, 2002, 2003]))
n_ny = ny_mask.sum()
for trgt_col, actl_col in trgt_actl_pairs:
    df.loc[ny_mask, trgt_col] = df.loc[ny_mask, actl_col]
print(f'NY SLRS (ppd_id=83): replaced targets with actuals for {n_ny} rows (FY 2001-2003)')

# Verify changes
idaho_check = df.loc[df['ppd_id'] == 31, ['fy', 'EQTotal_Trgt', 'PETotal_Trgt', 'EQTotal_Actl', 'PETotal_Actl']].head(5)
print('\nIdaho PERSI sample (should match actuals):')
print(idaho_check.to_string(index=False))

ny_check = df.loc[(df['ppd_id'] == 83) & (df['fy'].isin([2001, 2002, 2003])), 
                  ['fy', 'EQTotal_Trgt', 'FITotal_Trgt', 'EQTotal_Actl', 'FITotal_Actl']]
print('\nNY SLRS FY 2001-2003 (should match actuals):')
print(ny_check.to_string(index=False))

Idaho PERSI (ppd_id=31): replaced targets with actuals for 22 rows
NY SLRS (ppd_id=83): replaced targets with actuals for 3 rows (FY 2001-2003)

Idaho PERSI sample (should match actuals):
  fy  EQTotal_Trgt  PETotal_Trgt  EQTotal_Actl  PETotal_Actl
2001       0.66266       0.02402       0.66266       0.02402
2002       0.62937       0.02697       0.62937       0.02697
2003       0.65100       0.02500       0.65100       0.02500
2004       0.64800       0.02500       0.64800       0.02500
2005       0.63200       0.02800       0.63200       0.02800

NY SLRS FY 2001-2003 (should match actuals):
  fy  EQTotal_Trgt  FITotal_Trgt  EQTotal_Actl  FITotal_Actl
2001       0.54719       0.31887       0.54719       0.31887
2002       0.60415       0.29259       0.60415       0.29259
2003       0.54138       0.34171       0.54138       0.34171


## 3. Generate flag: trgt_zero_actl_nonzero

In [5]:
# 7 asset class pairs (excluding Cash and Other)
pairs = [
    ('EQTotal_Trgt', 'EQTotal_Actl'),
    ('FITotal_Trgt', 'FITotal_Actl'),
    ('RETotal_Trgt', 'RETotal_Actl'),
    ('AltMiscTotal_Trgt', 'AltMiscTotal_Actl'),
    ('PETotal_Trgt', 'PETotal_Actl'),
    ('HFTotal_Trgt', 'HFTotal_Actl'),
    ('COMDTotal_Trgt', 'COMDTotal_Actl'),
]

flag = pd.Series(False, index=df.index)
for trgt_col, actl_col in pairs:
    cond = (
        (df[trgt_col] == 0) &
        (df[actl_col] != 0) &
        df[trgt_col].notna() &
        df[actl_col].notna()
    )
    flag = flag | cond

df['trgt_zero_actl_nonzero'] = flag.astype(int)

print(f'trgt_zero_actl_nonzero distribution:')
print(df['trgt_zero_actl_nonzero'].value_counts().sort_index())
print(f'\nFinal shape: {df.shape[0]} rows x {df.shape[1]} cols')

trgt_zero_actl_nonzero distribution:
trgt_zero_actl_nonzero
0    5749
1     519
Name: count, dtype: int64

Final shape: 6268 rows x 284 cols


## 4. Save outputs

In [6]:
pyreadstat.write_dta(
    df,
    OUTPUT_PY / 'ppd_plan_clean_allocations.dta'
)
df.to_parquet(OUTPUT_PY / 'ppd_plan_clean_allocations.parquet', index=False)
df.to_csv(OUTPUT_PY / 'ppd_plan_clean_allocations.csv', index=False)

print('Saved: ppd_plan_clean_allocations.{dta, parquet, csv}')

Saved: ppd_plan_clean_allocations.{dta, parquet, csv}


## 5. Parity Check vs Stata Baseline

In [7]:
stata_df = pd.read_stata(OUTPUT_STATA / 'ppd_plan_clean_allocations.dta')
py_df = df.copy()

print('=== PARITY CHECK: ppd_plan_clean_allocations ===')
results = []

# 1. Shape
shape_ok = py_df.shape == stata_df.shape
results.append(('Shape', shape_ok))
print(f'\n1. Shape: Python={py_df.shape}, Stata={stata_df.shape} -> {"PASS" if shape_ok else "FAIL"}')

# 2. Column set
col_set_ok = set(py_df.columns) == set(stata_df.columns)
results.append(('Column set', col_set_ok))
print(f'2. Column set: {"PASS" if col_set_ok else "FAIL"}')
if not col_set_ok:
    only_py = sorted(set(py_df.columns) - set(stata_df.columns))
    only_st = sorted(set(stata_df.columns) - set(py_df.columns))
    if only_py: print(f'   Only in Python: {only_py}')
    if only_st: print(f'   Only in Stata:  {only_st}')

# 3. Column order
col_order_ok = list(py_df.columns) == list(stata_df.columns)
results.append(('Column order', col_order_ok))
print(f'3. Column order: {"PASS" if col_order_ok else "FAIL"}')

# 4. Null profile on allocation columns + flag
alloc_cols = sorted(set(r['col'] for r in replacements)) + ['trgt_zero_actl_nonzero']
null_ok = True
print(f'\n4. Null profiles (allocation cols + flag):')
for c in alloc_cols:
    if c in py_df.columns and c in stata_df.columns:
        pn = py_df[c].isna().sum()
        sn = stata_df[c].isna().sum()
        if pd.api.types.is_string_dtype(stata_df[c]):
            sn += (stata_df[c] == '').sum()
        if pd.api.types.is_string_dtype(py_df[c]):
            pn += (py_df[c] == '').sum()
        match = int(pn) == int(sn)
        if not match: null_ok = False
        print(f'   {c}: Py={pn}, St={sn} -> {"PASS" if match else "FAIL"}')
results.append(('Null profiles', null_ok))

# 5. Flag distribution
py_flag = py_df['trgt_zero_actl_nonzero'].value_counts().sort_index()
st_flag = stata_df['trgt_zero_actl_nonzero'].value_counts().sort_index()
flag_ok = py_flag.equals(st_flag)
results.append(('Flag distribution', flag_ok))
print(f'\n5. Flag distribution:')
print(f'   Python: {py_flag.to_dict()}')
print(f'   Stata:  {st_flag.to_dict()}')
print(f'   {"PASS" if flag_ok else "FAIL"}')

# Summary
print(f'\n{"=" * 50}')
for name, passed in results:
    print(f'  {"PASS" if passed else "FAIL"} - {name}')
print(f'{"=" * 50}')

=== PARITY CHECK: ppd_plan_clean_allocations ===

1. Shape: Python=(6268, 284), Stata=(6268, 284) -> PASS
2. Column set: PASS
3. Column order: PASS

4. Null profiles (allocation cols + flag):
   AltMiscTotal_Actl: Py=2166, St=2166 -> PASS
   AltMiscTotal_Trgt: Py=2486, St=2486 -> PASS
   COMDTotal_Actl: Py=2165, St=2165 -> PASS
   COMDTotal_Trgt: Py=2486, St=2486 -> PASS
   CashTotal_Actl: Py=2165, St=2165 -> PASS
   CashTotal_Trgt: Py=2487, St=2487 -> PASS
   EQTotal_Actl: Py=2164, St=2164 -> PASS
   EQTotal_Trgt: Py=2486, St=2486 -> PASS
   FITotal_Actl: Py=2164, St=2164 -> PASS
   FITotal_Trgt: Py=2486, St=2486 -> PASS
   HFTotal_Actl: Py=2165, St=2165 -> PASS
   HFTotal_Trgt: Py=2486, St=2486 -> PASS
   OtherTotal_Actl: Py=2165, St=2165 -> PASS
   OtherTotal_Trgt: Py=2486, St=2486 -> PASS
   PETotal_Actl: Py=2164, St=2164 -> PASS
   PETotal_Trgt: Py=2486, St=2486 -> PASS
   RETotal_Actl: Py=2165, St=2165 -> PASS
   RETotal_Trgt: Py=2486, St=2486 -> PASS
   trgt_zero_actl_nonzero: P

## 6. Deep Parity — Exhaustive Value Comparison

In [8]:
common_cols = sorted(set(py_df.columns) & set(stata_df.columns))

py_norm = py_df[common_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
stata_norm = stata_df[common_cols].sort_values(['ppd_id', 'fy']).reset_index(drop=True)

# Normalize empty strings -> NaN
for c in common_cols:
    if pd.api.types.is_string_dtype(stata_norm[c]):
        stata_norm[c] = stata_norm[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_norm[c]):
        py_norm[c] = py_norm[c].replace('', np.nan)

mismatches = {}
for col in common_cols:
    p = py_norm[col]
    s = stata_norm[col]
    nan_mismatch = int((p.isna() != s.isna()).sum())
    both_valid = p.notna() & s.notna()
    if both_valid.any():
        if pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s):
            pv = pd.to_numeric(p[both_valid], errors='coerce')
            sv = pd.to_numeric(s[both_valid], errors='coerce')
            val_mismatch = int((~np.isclose(pv, sv, rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
        else:
            val_mismatch = int((p[both_valid].astype(str) != s[both_valid].astype(str)).sum())
    else:
        val_mismatch = 0
    total = nan_mismatch + val_mismatch
    if total > 0:
        mismatches[col] = {'nan': nan_mismatch, 'val': val_mismatch, 'total': total}

if mismatches:
    print(f'MISMATCHES in {len(mismatches)} column(s):')
    for c, info in sorted(mismatches.items()):
        print(f'  {c}: nan_diff={info["nan"]}, val_diff={info["val"]}')
else:
    print(f'All {len(common_cols)} columns match across {len(py_norm)} rows.')

only_py = sorted(set(py_df.columns) - set(stata_df.columns))
only_st = sorted(set(stata_df.columns) - set(py_df.columns))
if only_py: print(f'\nOnly in Python: {only_py}')
if only_st: print(f'\nOnly in Stata: {only_st}')

overall = len(mismatches) == 0 and len(only_py) == 0 and len(only_st) == 0
print(f'\n{"EXACT PARITY CONFIRMED" if overall else "PARITY ISSUES DETECTED"}')

# Expected mismatches from Internet Appendix A.1.3 fixes (not in Stata code):
# - Idaho PERSI (ppd_id=31): 22 rows x target cols
# - NY SLRS (ppd_id=83, FY 2001-2003): 3 rows x target cols
# - trgt_zero_actl_nonzero flag may also differ for affected rows
if mismatches:
    expected_trgt_cols = {'AltMiscTotal_Trgt', 'COMDTotal_Trgt', 'CashTotal_Trgt',
                          'EQTotal_Trgt', 'FITotal_Trgt', 'HFTotal_Trgt',
                          'Leverage_Total_Trgt', 'OtherTotal_Trgt', 'PETotal_Trgt',
                          'RETotal_Trgt', 'trgt_zero_actl_nonzero'}
    unexpected = {c for c in mismatches if c not in expected_trgt_cols}
    if unexpected:
        print(f'\n*** WARNING: Unexpected mismatches in: {sorted(unexpected)}')
    else:
        print(f'\nAll mismatches are in expected target columns (Idaho PERSI + NY SLRS fixes). OK.')

MISMATCHES in 7 column(s):
  CashTotal_Trgt: nan_diff=0, val_diff=20
  EQTotal_Trgt: nan_diff=0, val_diff=25
  FITotal_Trgt: nan_diff=0, val_diff=25
  HFTotal_Trgt: nan_diff=0, val_diff=1
  PETotal_Trgt: nan_diff=0, val_diff=25
  RETotal_Trgt: nan_diff=0, val_diff=25
  trgt_zero_actl_nonzero: nan_diff=0, val_diff=24

PARITY ISSUES DETECTED

All mismatches are in expected target columns (Idaho PERSI + NY SLRS fixes). OK.


## 7. Joint Audit — Steps 1 through 3

Comprehensive cross-step validation: per-step parity, pipeline chain integrity, 
data flow consistency, and exhaustive value fidelity across all 3 stages.

In [9]:
# ── Load all outputs ──
py1 = pd.read_stata(OUTPUT_PY / 'mapping_table_fy_final.dta')
py2 = pd.read_stata(OUTPUT_PY / 'ppd_plan_level_raw.dta')
py3 = pd.read_stata(OUTPUT_PY / 'ppd_plan_clean_allocations.dta')
st1 = pd.read_stata(OUTPUT_STATA / 'mapping_table_fy_final.dta')
st2 = pd.read_stata(OUTPUT_STATA / 'ppd_plan_level_raw.dta')
st3 = pd.read_stata(OUTPUT_STATA / 'ppd_plan_clean_allocations.dta')

print('Loaded all 6 datasets (3 Python, 3 Stata)\n')

# ── Helper: exhaustive cell comparison ──
def compare_df(py, st, label):
    """Compare two DataFrames cell-by-cell. Returns (n_mismatched_cols, details_dict)."""
    common = sorted(set(py.columns) & set(st.columns))
    p = py[common].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
    s = st[common].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
    for c in common:
        if pd.api.types.is_string_dtype(s[c]):
            s[c] = s[c].replace('', np.nan)
        if pd.api.types.is_string_dtype(p[c]):
            p[c] = p[c].replace('', np.nan)
    bad = {}
    for c in common:
        pv, sv = p[c], s[c]
        nan_diff = int((pv.isna() != sv.isna()).sum())
        both = pv.notna() & sv.notna()
        if both.any():
            if pd.api.types.is_numeric_dtype(pv) or pd.api.types.is_numeric_dtype(sv):
                val_diff = int((~np.isclose(
                    pd.to_numeric(pv[both], errors='coerce'),
                    pd.to_numeric(sv[both], errors='coerce'),
                    rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
            else:
                val_diff = int((pv[both].astype(str) != sv[both].astype(str)).sum())
        else:
            val_diff = 0
        if nan_diff + val_diff > 0:
            bad[c] = {'nan': nan_diff, 'val': val_diff}
    return len(bad), bad

results = []
print('=' * 65)
print('  JOINT AUDIT — Steps 1 through 3')
print('=' * 65)

# ── CHECK 1-3: Shape parity per step ──
for i, (label, py, st) in enumerate([('Step 1', py1, st1), ('Step 2', py2, st2), ('Step 3', py3, st3)], 1):
    ok = py.shape == st.shape
    results.append((f'{i}. {label} shape', ok))
    print(f'\n {i}. {label} shape: Py={py.shape} St={st.shape} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 4-6: Exhaustive value comparison per step ──
for i, (label, py, st) in enumerate([('Step 1', py1, st1), ('Step 2', py2, st2), ('Step 3', py3, st3)], 4):
    n, det = compare_df(py, st, label)
    ok = n == 0
    results.append((f'{i}. {label} values', ok))
    print(f' {i}. {label} values: {f"{n} cols differ — {det}" if n else "all match"} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 7: Key uniqueness at every stage ──
dup1 = py1.duplicated(subset=['ppd_id', 'fy']).sum()
dup2 = py2.duplicated(subset=['ppd_id', 'fy']).sum()
dup3 = py3.duplicated(subset=['ppd_id', 'fy']).sum()
ok = dup1 == 0 and dup2 == 0 and dup3 == 0
results.append(('7. Key uniqueness (ppd_id, fy)', ok))
print(f' 7. Key uniqueness: Step1={dup1}, Step2={dup2}, Step3={dup3} dups -> {"PASS" if ok else "FAIL"}')

# ── CHECK 8: Step 1 -> Step 2 merge column propagation ──
step1_merge_cols = ['pub_id', 'pub_system_name', 'preqin_id', 'preqin_name']
present = [c for c in step1_merge_cols if c in py2.columns]
ok = len(present) == len(step1_merge_cols)
results.append(('8. Step 1->2 column propagation', ok))
print(f' 8. Step 1->2 column propagation: {len(present)}/{len(step1_merge_cols)} -> {"PASS" if ok else f"FAIL — missing: {set(step1_merge_cols)-set(present)}"}')

# ── CHECK 9: Step 2 -> Step 3 column preservation ──
s2_cols = set(py2.columns)
s3_cols = set(py3.columns)
new_in_s3 = s3_cols - s2_cols
missing_from_s3 = s2_cols - s3_cols
ok = missing_from_s3 == set() and new_in_s3 == {'trgt_zero_actl_nonzero'}
results.append(('9. Step 2->3 column preservation', ok))
print(f' 9. Step 2->3 columns: new={new_in_s3}, dropped={missing_from_s3} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 10: Non-allocation columns unchanged Step 2->3 ──
alloc_flag_cols = {
    'EQTotal_Actl', 'EQTotal_Trgt', 'FITotal_Actl', 'FITotal_Trgt',
    'CashTotal_Actl', 'CashTotal_Trgt', 'RETotal_Actl', 'RETotal_Trgt',
    'AltMiscTotal_Actl', 'AltMiscTotal_Trgt', 'PETotal_Actl', 'PETotal_Trgt',
    'HFTotal_Actl', 'HFTotal_Trgt', 'COMDTotal_Actl', 'COMDTotal_Trgt',
    'OtherTotal_Actl', 'OtherTotal_Trgt', 'trgt_zero_actl_nonzero'
}
non_alloc = sorted(s2_cols - alloc_flag_cols)
p2_sorted = py2[non_alloc].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
p3_sorted = py3[non_alloc].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
n_diff_na, det_na = 0, {}
for c in non_alloc:
    pv, sv = p2_sorted[c], p3_sorted[c]
    if pd.api.types.is_string_dtype(pv):
        pv = pv.replace('', np.nan)
    if pd.api.types.is_string_dtype(sv):
        sv = sv.replace('', np.nan)
    nan_d = int((pv.isna() != sv.isna()).sum())
    both = pv.notna() & sv.notna()
    if both.any() and pd.api.types.is_numeric_dtype(pv):
        val_d = int((~np.isclose(pv[both], sv[both], rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
    elif both.any():
        val_d = int((pv[both].astype(str) != sv[both].astype(str)).sum())
    else:
        val_d = 0
    if nan_d + val_d > 0:
        n_diff_na += 1
        det_na[c] = {'nan': nan_d, 'val': val_d}
ok = n_diff_na == 0
results.append(('10. Non-alloc cols unchanged Step 2->3', ok))
print(f'10. Non-alloc cols unchanged Step2->3: {f"{n_diff_na} diffs — {det_na}" if n_diff_na else "all match"} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 11: DTA round-trip fidelity ──
py3_reloaded = pd.read_stata(OUTPUT_PY / 'ppd_plan_clean_allocations.dta')
n, det = compare_df(py3_reloaded, st3, 'Roundtrip')
ok = n == 0
results.append(('11. DTA round-trip fidelity', ok))
print(f'11. DTA round-trip (save->reload): {f"{n} cols differ" if n else "all match"} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 12: Flag distribution match ──
py_flag_d = {int(k): v for k, v in py3['trgt_zero_actl_nonzero'].value_counts().sort_index().to_dict().items()}
st_flag_d = {int(k): v for k, v in st3['trgt_zero_actl_nonzero'].value_counts().sort_index().to_dict().items()}
ok = py_flag_d == st_flag_d
results.append(('12. Flag distribution', ok))
print(f'12. Flag distribution: Py={py_flag_d} St={st_flag_d} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 13: Row count consistency across pipeline ──
ok = py2.shape[0] == py3.shape[0]
results.append(('13. Row count Step 2 == Step 3', ok))
print(f'13. Row count Step2->Step3: {py2.shape[0]} == {py3.shape[0]} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 14: Column order matches Stata at each step ──
ok1 = list(py1.columns) == list(st1.columns)
ok2 = list(py2.columns) == list(st2.columns)
ok3 = list(py3.columns) == list(st3.columns)
ok = ok1 and ok2 and ok3
results.append(('14. Column order all steps', ok))
print(f'14. Column order: Step1={"Y" if ok1 else "N"} Step2={"Y" if ok2 else "N"} Step3={"Y" if ok3 else "N"} -> {"PASS" if ok else "FAIL"}')

# ── CHECK 15: Full null profile parity at each step ──
null_ok = True
null_issues = []
for label, py, st in [('Step1', py1, st1), ('Step2', py2, st2), ('Step3', py3, st3)]:
    common = sorted(set(py.columns) & set(st.columns))
    for c in common:
        pn = py[c].isna().sum()
        sn = st[c].isna().sum()
        if pd.api.types.is_string_dtype(st[c]):
            sn += (st[c] == '').sum()
        if pd.api.types.is_string_dtype(py[c]):
            pn += (py[c] == '').sum()
        if int(pn) != int(sn):
            null_ok = False
            null_issues.append(f'  {label}.{c}: Py={pn} St={sn}')
results.append(('15. Null profiles all steps', null_ok))
if null_issues:
    for issue in null_issues[:10]:
        print(issue)
print(f'15. Null profiles all steps -> {"PASS" if null_ok else "FAIL"}')

# ── SUMMARY ──
print(f'\n{"=" * 65}')
all_pass = True
for name, passed in results:
    tag = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f'  {tag} — {name}')
print(f'{"=" * 65}')
print(f'\n  {"ALL 15 CHECKS PASSED" if all_pass else "SOME CHECKS FAILED — SEE ABOVE"}')
print(f'  Steps 1->2->3 pipeline: {"VERIFIED" if all_pass else "NEEDS ATTENTION"}')

Loaded all 6 datasets (3 Python, 3 Stata)

  JOINT AUDIT — Steps 1 through 3

 1. Step 1 shape: Py=(5244, 15) St=(5244, 15) -> PASS

 2. Step 2 shape: Py=(6268, 283) St=(6268, 283) -> PASS

 3. Step 3 shape: Py=(6268, 284) St=(6268, 284) -> PASS
 4. Step 1 values: all match -> PASS
 5. Step 2 values: all match -> PASS
 6. Step 3 values: 7 cols differ — {'CashTotal_Trgt': {'nan': 0, 'val': 20}, 'EQTotal_Trgt': {'nan': 0, 'val': 25}, 'FITotal_Trgt': {'nan': 0, 'val': 25}, 'HFTotal_Trgt': {'nan': 0, 'val': 1}, 'PETotal_Trgt': {'nan': 0, 'val': 25}, 'RETotal_Trgt': {'nan': 0, 'val': 25}, 'trgt_zero_actl_nonzero': {'nan': 0, 'val': 24}} -> FAIL
 7. Key uniqueness: Step1=0, Step2=0, Step3=0 dups -> PASS
 8. Step 1->2 column propagation: 4/4 -> PASS
 9. Step 2->3 columns: new={'trgt_zero_actl_nonzero'}, dropped=set() -> PASS
10. Non-alloc cols unchanged Step2->3: all match -> PASS
11. DTA round-trip (save->reload): 7 cols differ -> FAIL
12. Flag distribution: Py={0: 5749, 1: 519} St={0: 5725,